# Practice 1. Building an analysis-ready data set

Industrial AI, KBTU. Goes with Lecture 2 and `inclass_work_2.ipynb`. **1.25 points.**

**The data.** Pump sensor data from Kaggle (`nphantawee/pump-sensor-data`): 52 sensors on a water
pump, one sample per minute, five months, seven breakdowns. The loader cell downloads it once.

The columns are `timestamp`, `sensor_00` … `sensor_51`, and `machine_status`, which is `NORMAL`,
`BROKEN` or `RECOVERING`. Nobody wrote down what each sensor measures. That is normal for a plant
historian, and it does not stop you from cleaning the file.

**The question.** Nobody tells you what is wrong with this file. Find it, decide what to do about
each problem, and end with a table that a diagnostic model could be trained on.

### How the tasks are built

Each task says what to write: the functions, their arguments and what they return, and sometimes
a variable with a result. How you write them is up to you, as long as you use the method the task
names. Under your cell sits a cell of **tests**. Run it after your code: it either prints
`all tests passed` or stops at the first check that failed and says what is wrong. The tests check
what your code does on small cases with known answers, not how it is written.

These tests are public. When you hand in, the same tests run again together with hidden ones, on
other inputs, and the share of tests passed gives the points for the code. Code that only fits the
public cases loses those points.

The cell marked *Common ground* is written for you. Run it as it is: it fixes the cleaned table
`work` and the list of sensors `FEATS`, so that everybody's numbers from Task 4 on can be
compared.

The methods are yours: the gap count, the run length, linear filling, the three outlier rules, the
hourly aggregates, the three scalers and both routes to PCA. pandas and numpy are there for the
plumbing (`groupby`, `diff`, `quantile`, `np.linalg.eigh`); `interpolate`, `np.interp`,
`sklearn.preprocessing` and `sklearn.decomposition.PCA` are for checking your result only.

### What you hand in

Two files:

1. this notebook, run top to bottom, named `Surname_Name_practice_1.ipynb`;
2. `practice_1_answers.md` with your answers to the questions under each task, named
   `Surname_Name_practice_1_answers.md`.

Every answer needs the numbers your notebook printed. A number the notebook does not print scores
zero, and so does a number printed with nothing said about it.

| Task | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | total |
|---|---|---|---|---|---|---|---|---|---|
| code (tests) | 0.04 | 0.05 | 0.10 | 0.10 | 0.06 | 0.08 | 0.10 | 0.07 | 0.60 |
| answers | 0.06 | 0.05 | 0.10 | 0.10 | 0.09 | 0.07 | 0.10 | 0.08 | 0.65 |

### The rules

- Every decision needs a reason. "I dropped `sensor_15`" is not an answer. "`sensor_15` is
  empty in all rows, so it carries nothing and I dropped it" is.
- Do not delete rows you cannot justify deleting. You are looking for breakdowns, and a
  breakdown is an outlier.
- Do not rename the functions and variables the tests use, and do not edit the test cells.
- Before handing in, restart the kernel and run all cells; the whole notebook runs in about a
  minute on a laptop. Then check your answers against the new output.
- Figures need axis labels, units and a title that says what to look at.
- Stuck? `inclass_work_2.ipynb` does every one of these steps on a smaller data set, and
  `handout.ipynb` has the patterns on one page.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

import sklearn

BLUE, ORANGE, RED, GREEN, PURPLE, GREY = (
    "#2a6f8e", "#e07b39", "#c0392b", "#4a8f5a", "#9b3f9b", "#9aa0a6")

mpl.rcParams.update({
    "figure.dpi": 110,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.edgecolor": "#cccccc",
    "axes.grid": True,
    "axes.axisbelow": True,
    "grid.color": "#e6e6e6",
    "grid.linewidth": 0.8,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "axes.titlelocation": "left",
    "axes.labelsize": 10,
    "font.size": 10,
    "font.family": ["Segoe UI", "DejaVu Sans", "sans-serif"],
    "legend.frameon": False,
    "lines.linewidth": 1.6,
})

pd.set_option("display.width", 130)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

KAGGLE = {
    'pump_sensor': 'nphantawee/pump-sensor-data',
}


def dataset_folder(name):
    """The folder with the files of a data set: the Kaggle copy (downloaded once, then cached),
    or data/<name>/ of the course repo if Kaggle is out of reach."""
    if KAGGLE[name]:
        try:
            import contextlib, io, logging, kagglehub
            logging.getLogger("kagglehub").setLevel(logging.ERROR)
            with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
                folder = Path(kagglehub.dataset_download(KAGGLE[name]))
            print(f"{name}: Kaggle {KAGGLE[name]}")
            return folder
        except Exception as exc:
            print(f"Kaggle unavailable ({type(exc).__name__}), looking for data/{name}/")
    for root in [Path.cwd(), *Path.cwd().parents]:
        if (root / "data" / name).is_dir():
            return root / "data" / name
    raise FileNotFoundError(f"no data/{name}/ here or above; run data/fetch.py in the course repo")

print(" | ".join(f"{m} {sys.modules[m].__version__}" for m in
                 ("numpy", "pandas", "matplotlib", "scipy", "sklearn", "torch") if m in sys.modules))

numpy 2.5.5 | pandas 3.1.1 | matplotlib 3.11.2 | scipy 1.18.1 | sklearn 1.9.1


In [2]:
def load_pump():
    path = next(dataset_folder("pump_sensor").rglob("*.csv"))       # sensor.csv
    df = pd.read_csv(path, parse_dates=["timestamp"])
    return df.drop(columns=[c for c in df.columns if c.startswith("Unnamed")])


pump = load_pump()
SENSORS = [c for c in pump.columns if c.startswith("sensor_")]
print(pump.shape, "|", len(SENSORS), "sensors")

pump_sensor: Kaggle nphantawee/pump-sensor-data


(220320, 54) | 52 sensors


---

## Task 1. What is in the file  
*0.10 points: 0.04 for the code (tests), 0.06 for the answers*

Take inventory before you touch anything. A plant historian logs on a fixed clock, so the first
question is whether the clock is complete. With timestamps $t_1, \dots, t_m$ and a step
$\Delta$, the regular grid runs from the earliest to the latest timestamp,

$$g_k = t_{\min} + k\Delta, \qquad k = 0, 1, \dots, \left\lfloor \frac{t_{\max} - t_{\min}}{\Delta} \right\rfloor,$$

and every grid point that no timestamp falls on is a missing sample. Counting rows is not enough:
a duplicated row hides a hole of the same size.

**Write**

- `count_missing(ts, step)`: the number of grid points with no timestamp, as an `int`. `ts` is a
  sequence of timestamps (a `pd.Series`, a list, an index) in any order, possibly with duplicates;
  `step` is a pandas frequency such as `"1min"` or a `pd.Timedelta`;
- `status_counts`: a `pd.Series`, the number of rows of `pump` with each `machine_status`;
- `breakdown_times`: a list of the `pd.Timestamp`s of the rows with status `BROKEN`, in time
  order.

In [ ]:
def count_missing(ts, step):
    ...


status_counts = ...
breakdown_times = ...

In [ ]:
# Tests for Task 1 (public). They check what your code does, not how it is written.
# Run this cell after your code; an AssertionError says which check failed.


def test_three_holes():
    minutes = ["00:00", "00:01", "00:03", "00:06"]
    ts = pd.to_datetime([f"2020-01-01 {m}" for m in minutes])
    got = count_missing(ts, "1min")
    is_int = isinstance(got, (int, np.integer))
    assert is_int, f"count_missing returns an int, got {type(got).__name__}"
    assert got == 3, f"7 grid points 00:00..00:06, 4 present: expected 3, got {got}"


test_three_holes()


def test_order_and_duplicates_do_not_matter():
    minutes = ["00:06", "00:00", "00:03", "00:03", "00:01"]
    ts = pd.Series(pd.to_datetime([f"2020-01-01 {m}" for m in minutes]))
    got = count_missing(ts, "1min")
    assert got == 3, "sort the timestamps and count a duplicate once"
    got = count_missing(ts, pd.Timedelta("3min"))
    assert got == 0, "with a 3-minute step 00:00, 00:03, 00:06 are all there"


test_order_and_duplicates_do_not_matter()


def test_results_have_the_right_type():
    message = "status_counts: a Series indexed by the three statuses"
    assert isinstance(status_counts, pd.Series), message
    assert set(status_counts.index) == {"NORMAL", "BROKEN", "RECOVERING"}, message

    message = "breakdown_times: a list of pd.Timestamp"
    assert isinstance(breakdown_times, list), message
    assert all(isinstance(t, pd.Timestamp) for t in breakdown_times), message


test_results_have_the_right_type()


print("all tests passed")

**Questions** (answer them in `practice_1_answers.md`, Task 1, with the numbers your code printed):

1. Report the shape of the table, the time span, the sampling step and the number of missing grid points. Is the clock regular?
2. How many rows have each status, and when were the breakdowns? How much memory does the table take?
3. Does `BROKEN` mark a period of failure or a single moment? What share of the rows are breakdowns, and what does that share mean for a model judged by accuracy?

In [ ]:
# your code for the questions: tables, figures, anything you quote in the answers

---

## Task 2. Columns and rows that carry nothing  
*0.10 points: 0.05 for the code (tests), 0.05 for the answers*

A stuck tag is one that stops changing. The longest run of identical consecutive values finds
it. A standard way: mark where the value changes and number the runs with a running sum,

$$c_i = [\,v_i \ne v_{i-1}\,], \qquad g_i = \sum_{j \le i} c_j .$$

$c_i$ is 1 where a new run starts and 0 inside a run, so $g$ gives every run its own id:
`1, 1, 1, 2, 2, 3, ...`. Group by the id and take the size of the largest group. A missing value
is not a value: it breaks a run and never belongs to one.

**Write**

- `longest_run(v)`: the length of the longest run of identical consecutive non-missing values
  of a `pd.Series`, as an `int`; 0 if `v` has no values at all;
- `empty_sensors`: a list of the sensors of `pump` that have no value in any row;
- `n_repeated`: an `int`, the number of rows whose 52 sensor readings repeat an earlier row
  exactly (missing values count as equal, as in `duplicated`);
- `longest`: a `pd.Series` indexed by `SENSORS`, the longest run of each sensor, from your
  function.

In [ ]:
def longest_run(v):
    ...


empty_sensors = ...
n_repeated = ...
longest = ...

In [ ]:
# Tests for Task 2 (public). They check what your code does, not how it is written.
# Run this cell after your code; an AssertionError says which check failed.


def test_runs_of_integers():
    got = longest_run(pd.Series([1, 1, 1, 2, 2, 3, 3, 3, 3, 3, 4]))
    assert got == 5, f"the run of five 3s is the longest, got {got}"
    assert longest_run(pd.Series([7, 7, 7, 7, 8])) == 4
    assert isinstance(got, (int, np.integer)), "return an int"


test_runs_of_integers()


def test_missing_values_break_a_run():
    got = longest_run(pd.Series([1.0, 1.0, np.nan, 1.0, 1.0, 1.0]))
    assert got == 3, f"the NaN splits the 1s into runs of 2 and 3, got {got}"
    all_missing = pd.Series([np.nan, np.nan, np.nan])
    assert longest_run(all_missing) == 0, "a series with no values has no run"


test_missing_values_break_a_run()


def test_results_have_the_right_type():
    message = "empty_sensors: a list of sensor names"
    assert isinstance(empty_sensors, list), message
    assert set(empty_sensors) <= set(SENSORS), message

    assert isinstance(n_repeated, (int, np.integer)), "n_repeated: an int"

    message = "longest: a Series indexed by SENSORS"
    assert isinstance(longest, pd.Series), message
    assert list(longest.index) == SENSORS, message


test_results_have_the_right_type()


print("all tests passed")

**Questions** (answer them in `practice_1_answers.md`, Task 2, with the numbers your code printed):

1. Which sensor is empty? How many rows repeat an earlier reading, as a count and as a share? Which five sensors have the longest constant run, in hours?
2. Would you drop the repeated rows? Look at the timestamps of Task 1 before you answer.
3. What does a constant run of several days say about a tag on a working pump, and what would you do with those sensors before modelling?

In [ ]:
# your code for the questions: tables, figures, anything you quote in the answers

---

## Task 3. Gaps  
*0.20 points: 0.10 for the code (tests), 0.10 for the answers*

A gap is a run of missing values. How long the runs are decides what filling them means: one
missing minute between two readings is safe to bridge, a block of six hours is invented data.

**Linear filling by position.** Let a gap lie between the known values $x_a$ and $x_b$ at row
positions $a < b$. Inside it,

$$x_k = x_a + \frac{k - a}{b - a}\,(x_b - x_a), \qquad a < k < b,$$

the straight line between the two neighbours. Before the first known value and after the last
there is only one neighbour, so those rows take the nearest known value. The positions are row
numbers, not timestamps: the clock of this file is regular, so the two agree.

**Write**

- `gap_lengths(s)`: a 1-D integer `np.ndarray`, the length of every run of missing values in the
  `pd.Series` `s`, in the order the runs appear; empty if there is no gap;
- `fill_linear(s)`: a new `pd.Series` with the same index and name as `s` and every gap filled by
  the rule above; `s` itself stays unchanged, and a series with no values at all comes back as
  it is (all missing). Write the rule yourself; `interpolate` and `np.interp` are for checking;
- `missing_pct`: a `pd.Series` indexed by sensor, the percentage of missing rows of each of the 52
  sensors (0 to 100), largest first.

In [ ]:
def gap_lengths(s):
    ...


def fill_linear(s):
    ...


missing_pct = ...

In [ ]:
# Tests for Task 3 (public). They check what your code does, not how it is written.
# Run this cell after your code; an AssertionError says which check failed.


def test_two_gaps():
    got = gap_lengths(pd.Series([1.0, np.nan, np.nan, 4.0, np.nan, 6.0]))
    assert isinstance(got, np.ndarray), "gap_lengths returns a numpy array"
    assert got.tolist() == [2, 1], f"a gap of 2, then a gap of 1; got {got.tolist()}"


test_two_gaps()


def test_gaps_at_the_ends_and_no_gaps():
    at_ends = gap_lengths(pd.Series([np.nan, 1.0, np.nan]))
    assert at_ends.tolist() == [1, 1], "gaps at both ends count"
    assert len(gap_lengths(pd.Series([1.0, 2.0]))) == 0, "no gap: an empty array"


test_gaps_at_the_ends_and_no_gaps()


def test_straight_line_inside_a_gap():
    got = fill_linear(pd.Series([1.0, np.nan, np.nan, 4.0]))
    assert isinstance(got, pd.Series), "fill_linear returns a Series"
    expected = [1, 2, 3, 4]
    filled = got.to_numpy()
    assert np.allclose(filled, expected), f"expected {expected}, got {got.tolist()}"


test_straight_line_inside_a_gap()


def test_ends_take_the_nearest_value():
    s = pd.Series([np.nan, 2.0, np.nan, np.nan, 5.0, np.nan], index=list("abcdef"))
    got = fill_linear(s)
    expected = [2, 2, 3, 4, 5, 5]
    filled = got.to_numpy()
    assert np.allclose(filled, expected), f"expected {expected}, got {got.tolist()}"
    assert list(got.index) == list("abcdef"), "keep the index of s"
    assert s.isna().sum() == 4, "s itself must stay unchanged"


test_ends_take_the_nearest_value()


def test_missing_pct_shape():
    assert isinstance(missing_pct, pd.Series), "one value per sensor"
    assert set(missing_pct.index) == set(SENSORS), "one value per sensor"
    assert missing_pct.is_monotonic_decreasing, "largest first"
    assert 0 <= missing_pct.min() and missing_pct.max() <= 100, "a percentage, 0 to 100"


test_missing_pct_shape()


print("all tests passed")

*Common ground: run this cell as it is, after your tests pass.*

In [ ]:
# The cleaned table for Tasks 4 to 8, the same for everybody. Your own choice in Task 3
# may differ: argue it in the answers, but keep this cell as it is.
na_share = pump[SENSORS].isna().mean()
DROP = list(na_share[na_share > 0.50].index)                  # nothing left to model
FLAG = list(na_share[(na_share > 0.05) & (na_share <= 0.50)].index)
KEEP = [c for c in SENSORS if c not in DROP]

work = pump.drop(columns=DROP).copy()
for c in FLAG:
    work[c + "_was_missing"] = work[c].isna().astype(int)     # the gap itself is data
work[KEEP] = work[KEEP].interpolate(method="linear", limit_direction="both")
FEATS = KEEP                                                  # the sensors from here on

print("dropped:", DROP, "| flagged:", FLAG, "| sensors kept:", len(FEATS),
      "| gaps left:", int(work[FEATS].isna().sum().sum()))

**Questions** (answer them in `practice_1_answers.md`, Task 3, with the numbers your code printed):

1. Report the percentage missing for the worst ten sensors and the number of sensors under 0.1 %. Draw a missingness map (sensors against rows).
2. Run `gap_lengths` on `sensor_50`, `sensor_51` and `sensor_00`: the number of gaps, the longest gap in hours, and the share of missing rows that sit in gaps longer than an hour. Are the gaps scattered samples or blocks?
3. Split the sensors into three groups: drop the column, keep and flag, leave alone. Say where you put each boundary and why. Compare your choice with the common cell below.
4. `sensor_50` misses about a third of its values. Would you trust a model that uses it? Back the answer with your numbers from question 2.

In [ ]:
# your code for the questions: tables, figures, anything you quote in the answers

---

## Task 4. Outliers, and whether they are errors  
*0.20 points: 0.10 for the code (tests), 0.10 for the answers*

Three rules. Each flags a value $x$ that falls outside two fences.

**3-sigma.** With $\mu$ the mean and $\sigma$ the sample standard deviation ($n - 1$ in the
denominator, the pandas default):

$$[\;\mu - k\sigma,\;\; \mu + k\sigma\;], \qquad k = 3.$$

**IQR (Tukey).** With $Q_1$ and $Q_3$ the 25th and 75th percentiles (linear interpolation between
order statistics, the pandas default) and $\mathrm{IQR} = Q_3 - Q_1$:

$$[\;Q_1 - k\cdot\mathrm{IQR},\;\; Q_3 + k\cdot\mathrm{IQR}\;], \qquad k = 1.5.$$

**Robust z-score.** With $m$ the median and $\mathrm{MAD} = \mathrm{median}\,|x - m|$,

$$z = \frac{0.6745\,(x - m)}{\mathrm{MAD}}, \qquad \text{flag } |z| > k, \quad k = 3.5,$$

so the fences are $m \pm k\,\mathrm{MAD}/0.6745$.

For normal data $\mathrm{MAD} = 0.6745\,\sigma$, so the factor makes $z$ agree with an ordinary
z-score, and the threshold 3.5 means about 3.5 standard deviations. On normal data the 3-sigma
band leaves out $2\,(1 - \Phi(3)) = 0.27\,\%$. The quartiles are $\pm 0.674\sigma$, so the IQR
fences sit at $\pm(0.674 + 1.5 \cdot 1.349)\sigma = \pm 2.70\sigma$ and leave out 0.70 %. The
difference is in what sets the fences: $\mu$ and $\sigma$ move with every extreme value, the
quartiles and the median do not.

**Write**

- `three_sigma(s, k=3.0)`, `iqr_rule(s, k=1.5)`, `robust_z(s, k=3.5)`: each takes a `pd.Series`
  and returns `(mask, (low, high))`. `mask` is a boolean `pd.Series` with the index of `s`,
  `True` where the value lies strictly outside the fences; `low` and `high` are the fences as
  floats. Missing values are skipped by the statistics and never flagged;
- `iqr_recovering`: a `pd.Series` indexed by `FEATS`: for each sensor of `work`, the share
  (0 to 1) of `RECOVERING` among the rows that `iqr_rule` flags; `NaN` where it flags none.

In [ ]:
def three_sigma(s, k=3.0):
    ...


def iqr_rule(s, k=1.5):
    ...


def robust_z(s, k=3.5):
    ...


iqr_recovering = ...

In [ ]:
# Tests for Task 4 (public). They check what your code does, not how it is written.
# Run this cell after your code; an AssertionError says which check failed.


def test_fences_on_eight_values():
    s = pd.Series([2.0, 4, 4, 4, 5, 5, 7, 9])
    m3, (lo3, hi3) = three_sigma(s)
    half_width = 3 * np.sqrt(32 / 7)
    fences_ok = np.isclose(lo3, 5 - half_width) and np.isclose(hi3, 5 + half_width)
    assert fences_ok, f"mean 5, sd sqrt(32/7) with n - 1; got fences {lo3:.3f}, {hi3:.3f}"

    mi, (loi, hii) = iqr_rule(s)
    fences_ok = np.isclose(loi, 1.75) and np.isclose(hii, 7.75)
    assert fences_ok, f"Q1 = 4, Q3 = 5.5: fences 1.75 and 7.75, got {loi}, {hii}"
    assert mi.tolist() == [False] * 7 + [True], "only 9 lies outside the IQR fences"

    mr, (lor, hir) = robust_z(s)
    half_width = 3.5 * 0.5 / 0.6745
    fences_ok = np.isclose(lor, 4.5 - half_width) and np.isclose(hir, 4.5 + half_width)
    assert fences_ok, f"median 4.5, MAD 0.5; got fences {lor:.4f}, {hir:.4f}"


test_fences_on_eight_values()


def test_masks_are_boolean_series():
    s = pd.Series([1.0, 2.0, 3.0, 50.0], index=[10, 20, 30, 40])
    for rule in (three_sigma, iqr_rule, robust_z):
        mask, fences = rule(s)
        is_bool_series = isinstance(mask, pd.Series) and mask.dtype == bool
        assert is_bool_series, f"{rule.__name__}: the mask is a boolean Series"
        index_kept = list(mask.index) == [10, 20, 30, 40]
        assert index_kept, f"{rule.__name__}: the mask keeps the index of s"


test_masks_are_boolean_series()


def test_share_flagged_on_normal_data():
    t = pd.Series(np.random.default_rng(0).normal(0, 1, 200_000))
    share = three_sigma(t)[0].mean()
    assert abs(share - 0.0027) < 0.0006, "3-sigma flags about 0.27 % of normal data"
    share = iqr_rule(t)[0].mean()
    assert abs(share - 0.0070) < 0.0008, "the IQR rule flags about 0.70 % of normal data"
    share = robust_z(t)[0].mean()
    assert share < 0.001, "the robust z-score flags well under 0.1 % of normal data"


test_share_flagged_on_normal_data()


def test_iqr_recovering_shape():
    assert isinstance(iqr_recovering, pd.Series), "one value per sensor of FEATS"
    assert list(iqr_recovering.index) == FEATS, "one value per sensor of FEATS"
    v = iqr_recovering.dropna()
    assert ((v >= 0) & (v <= 1)).all(), "a share, 0 to 1"


test_iqr_recovering_shape()


print("all tests passed")

**Questions** (answer them in `practice_1_answers.md`, Task 4, with the numbers your code printed):

1. Pick three sensors: the most symmetric one, the most skewed one, and `sensor_51`. For each, report the skew and the number of samples that 3-sigma, IQR and the robust z-score flag, and draw its histogram with both sets of fences on a log y-axis.
2. What share of the whole file is `RECOVERING`? Report `iqr_recovering` for your three sensors and the five sensors with the highest share. What do the flagged samples have in common?
3. Would you delete the flagged samples? Justify it with your numbers.

In [ ]:
# your code for the questions: tables, figures, anything you quote in the answers

---

## Task 5. Resampling  
*0.15 points: 0.06 for the code (tests), 0.09 for the answers*

One sample a minute is more than a diagnostic model needs, and it is a lot of rows. Grouping the
minutes into clock hours keeps a few numbers per hour. For the samples $x_1, \dots, x_m$ of one
hour,

$$\bar x = \frac1m \sum_i x_i, \qquad \max_i x_i, \qquad s = \sqrt{\frac{1}{m-1}\sum_i (x_i - \bar x)^2}.$$

The mean follows the slow level, the maximum keeps short peaks, the standard deviation shows how
restless the hour was. An event shorter than an hour survives only in the last two.

**Write**

- `hourly(s)`: for a `pd.Series` with a `DatetimeIndex`, a `pd.DataFrame` with the columns
  `"mean"`, `"max"`, `"std"` (with $m - 1$), one row per clock hour that holds at least one
  non-missing sample, indexed by the start of the hour (00:00, 01:00, ...). Hours without samples
  do not appear; an hour with one sample has `std` NaN;
- `hourly_04`: `hourly` of `sensor_04` of `work`, with `timestamp` as the index.

In [ ]:
def hourly(s):
    ...


hourly_04 = ...

In [ ]:
# Tests for Task 5 (public). They check what your code does, not how it is written.
# Run this cell after your code; an AssertionError says which check failed.


def test_three_samples_in_two_hours():
    times = pd.to_datetime(["2020-01-01 00:10", "2020-01-01 00:50", "2020-01-01 02:30"])
    s = pd.Series([1.0, 3.0, 5.0], index=times)
    got = hourly(s)
    assert isinstance(got, pd.DataFrame), "columns mean, max, std"
    assert list(got.columns) == ["mean", "max", "std"], "columns mean, max, std"

    hours = list(pd.to_datetime(["2020-01-01 00:00", "2020-01-01 02:00"]))
    message = ("one row per hour with samples, indexed by the start of the hour; "
               "01:00 has none")
    assert list(got.index) == hours, message
    assert np.allclose(got["mean"], [2, 5]) and np.allclose(got["max"], [3, 5])

    std = got["std"]
    assert np.isclose(std.iloc[0], np.sqrt(2)) and np.isnan(std.iloc[1]), "std with n - 1"


test_three_samples_in_two_hours()


def test_hourly_04_shape():
    assert isinstance(hourly_04, pd.DataFrame)
    assert list(hourly_04.columns) == ["mean", "max", "std"]

    index = hourly_04.index
    assert isinstance(index, pd.DatetimeIndex), "indexed by whole hours"
    assert (index.minute == 0).all(), "indexed by whole hours"


test_hourly_04_shape()


print("all tests passed")

**Questions** (answer them in `practice_1_answers.md`, Task 5, with the numbers your code printed):

1. Plot the hourly mean, max and std of `sensor_04` over the 48 hours around one of the breakdowns, with the breakdown marked. Quote the three aggregates for the day before the break and for the hour of the break.
2. Which aggregate shows the failure coming, which hides it, and which only marks it? If none shows it coming, say so, and check a second breakdown before you conclude.
3. What did going from minutes to hours cost? Name an event this table can no longer show.

In [ ]:
# your code for the questions: tables, figures, anything you quote in the answers

---

## Task 6. Scaling  
*0.15 points: 0.08 for the code (tests), 0.07 for the answers*

PCA in the next task believes whichever sensor has the biggest numbers, so the columns have to be
put on one scale first. Three scalers, each applied to one column $\mathbf{x}$ at a time:

$$\text{min-max: } x' = \frac{x - x_{\min}}{x_{\max} - x_{\min}}, \qquad
\text{z-score: } x' = \frac{x - \mu}{\sigma}, \qquad
\text{robust: } x' = \frac{x - m}{Q_3 - Q_1}.$$

Here $\sigma$ is the population standard deviation ($n$ in the denominator, the numpy default),
$m$ the median, $Q_1$ and $Q_3$ the quartiles (`np.percentile`). Note where each one gets its two
numbers. Min-max takes them from the two most extreme values of the column, so a single stuck
register sets the whole scale. The z-score takes them from all values, the robust scaler from the
middle half only.

**Write**

- `minmax(x)`, `zscore(x)`, `robust(x)`: each takes a 1-D array, or a 2-D array whose columns are
  scaled separately, and returns a float `np.ndarray` of the same shape; `x` itself stays
  unchanged;
- `broken_scaling`: a `pd.DataFrame` with the index `["min-max", "z-score"]` and the columns
  `["median before", "IQR before", "median after", "IQR after"]`. Take `sensor_04` of `work` as a
  float array, scale it, and report the median and the IQR ($Q_3 - Q_1$) of the scaled column;
  then set its first value to `32767`, a stuck Modbus register, and do the same again.

In [ ]:
def minmax(x):
    ...


def zscore(x):
    ...


def robust(x):
    ...


broken_scaling = ...

In [ ]:
# Tests for Task 6 (public). They check what your code does, not how it is written.
# Run this cell after your code; an AssertionError says which check failed.


def test_five_values():
    t = np.array([10., 20., 30., 40., 100.])
    got = minmax(t)
    assert np.allclose(got, [0, 1 / 9, 2 / 9, 3 / 9, 1]), f"min-max: got {got}"

    got = zscore(t)
    expected = (t - 40) / np.sqrt(1000)
    assert np.allclose(got, expected), f"z-score, mean 40 and sd sqrt(1000): got {got}"

    got = robust(t)
    expected = [-1, -0.5, 0, 0.5, 3.5]
    assert np.allclose(got, expected), f"robust with median 30 and IQR 20: got {got}"


test_five_values()


def test_columns_are_scaled_separately():
    t = np.array([10., 20., 30., 40., 100.])
    M = np.column_stack([t, -3 * t + 7])
    for f in (minmax, zscore, robust):
        got = f(M)
        same_shape = isinstance(got, np.ndarray) and got.shape == M.shape
        assert same_shape, f"{f.__name__}: the same shape as the input"
        assert np.allclose(got[:, 0], f(t)), f"{f.__name__}: each column on its own"

    z = zscore(M)
    message = "z-score: mean 0 and sd 1 in every column"
    assert np.allclose(z.mean(0), 0) and np.allclose(z.std(0), 1), message


test_columns_are_scaled_separately()


def test_broken_scaling_shape():
    assert isinstance(broken_scaling, pd.DataFrame)
    assert list(broken_scaling.index) == ["min-max", "z-score"]
    columns = ["median before", "IQR before", "median after", "IQR after"]
    assert list(broken_scaling.columns) == columns


test_broken_scaling_shape()


print("all tests passed")

**Questions** (answer them in `practice_1_answers.md`, Task 6, with the numbers your code printed):

1. Box-plot the first ten sensors of `FEATS` raw, after min-max and after z-score. What does the raw panel hide?
2. Report `broken_scaling`. What did one bad value out of all the rows do to each scaler?
3. Which scaler would you take into Task 7, and why? Give two reasons.

In [ ]:
# your code for the questions: tables, figures, anything you quote in the answers

---

## Task 7. PCA  
*0.20 points: 0.10 for the code (tests), 0.10 for the answers*

$\mathbf{X}$ is the $n \times p$ matrix of sensor readings, one row per minute, one column per
sensor.

**1. Standardise** each column (population standard deviation, as in `zscore`); otherwise PCA
describes the sensor with the biggest units: $Z_{ij} = (X_{ij} - \mu_j)/\sigma_j$.

**2. Covariance matrix** of the standardised data, $p \times p$:
$\mathbf{C} = \frac{1}{n-1}\mathbf{Z}^{\top}\mathbf{Z}$.

**3. Eigen-decomposition** $\mathbf{C}\mathbf{v}_i = \lambda_i \mathbf{v}_i$. $\mathbf{C}$ is
symmetric, so use `np.linalg.eigh`, which returns real eigenvalues and orthonormal eigenvectors.

**4. Sort** in descending order of $\lambda$: `eigh` returns them smallest first.

**5. Share of variance** of component $i$: $r_i = \lambda_i / \sum_j \lambda_j$.

**6. Project** onto the components: $\mathbf{P} = \mathbf{Z}\mathbf{V}$, with the eigenvectors
as the columns of $\mathbf{V}$. Column $i$ of $\mathbf{P}$ is the score of every row on component
$i$.

Why the eigenvectors? The variance of the projection $\mathbf{Z}\mathbf{w}$ onto a unit vector
$\mathbf{w}$ is $\mathbf{w}^\top\mathbf{C}\mathbf{w}$. Maximise it subject to
$\mathbf{w}^\top\mathbf{w} = 1$ with a Lagrange multiplier and the condition is
$\mathbf{C}\mathbf{w} = \lambda\mathbf{w}$: the best direction is the top eigenvector, and the
variance it keeps is its eigenvalue. So the variance of score column $i$ is exactly $\lambda_i$.
The sign of each eigenvector is arbitrary: $-\mathbf{v}_i$ is as good as $\mathbf{v}_i$.

**Write**

- `pca_by_hand(X)`: for an $n \times p$ array returns `(lam, vec, ratio, scores)`: the $p$
  eigenvalues in descending order, the $p \times p$ matrix with the eigenvectors as columns in the
  same order, the $p$ shares of variance, and the $n \times p$ scores;
- `k90`: an `int`, the smallest number of components of `work[FEATS]` whose shares add up to at
  least 90 %;
- `share_2`: a `float`, the share of variance (0 to 1) of the first two components of
  `work[FEATS]`.

In [ ]:
def pca_by_hand(X):
    ...


k90 = ...
share_2 = ...

In [ ]:
# Tests for Task 7 (public). They check what your code does, not how it is written.
# Run this cell after your code; an AssertionError says which check failed.


def test_ratios_agree_with_sklearn():
    from sklearn.decomposition import PCA
    mixing = np.array([[3, 1, 0, 0], [0, 2, 0, 0], [0, 0, 1, 0], [0, 0, 0, 0.5]])
    demo = np.random.default_rng(0).normal(size=(500, 4)) @ mixing
    lam, vec, ratio, scores = pca_by_hand(demo)
    ref = PCA().fit((demo - demo.mean(0)) / demo.std(0))
    ref_ratio = ref.explained_variance_ratio_
    assert np.allclose(ratio, ref_ratio), (
        f"ratios {np.round(ratio, 4)}, sklearn {ref_ratio.round(4)}")


test_ratios_agree_with_sklearn()


def test_eigenvalues_sorted_and_scores_carry_them():
    demo = np.random.default_rng(1).normal(size=(300, 3)) * [1, 5, 2] + [0, 100, -3]
    demo[:, 2] += demo[:, 0]
    lam, vec, ratio, scores = pca_by_hand(demo)
    assert np.all(np.diff(lam) <= 0), "eigenvalues in descending order"
    assert scores.shape == (300, 3) and vec.shape == (3, 3), "scores n x p, vec p x p"
    assert np.allclose(vec.T @ vec, np.eye(3)), "the eigenvectors are orthonormal columns"
    score_var = scores.var(axis=0, ddof=1)
    assert np.allclose(score_var, lam), "the variance of score column i is lambda_i"


test_eigenvalues_sorted_and_scores_carry_them()


def test_results_have_the_right_type():
    message = "k90: a number of components"
    assert isinstance(k90, (int, np.integer)) and 1 <= k90 <= len(FEATS), message
    message = "share_2: a share between 0 and 1"
    assert isinstance(share_2, float) and 0 < share_2 <= 1, message


test_results_have_the_right_type()


print("all tests passed")

**Questions** (answer them in `practice_1_answers.md`, Task 7, with the numbers your code printed):

1. How many components do you need for 90 % of the variance, and what share do the first two keep? Draw the scree plot with the cumulative curve on it.
2. Project onto the first two components and plot all the minutes, coloured by `machine_status`, `NORMAL` first, small and faint. Report the mean position of each status on PC1 and PC2. Does the projection separate the pump's states?
3. Name the five sensors that weigh most in PC1 and in PC2. How far apart are the first two eigenvalues, and what does that say about naming the components?
4. In the in-class work two components of AI4I kept 75 %. Why does this data set need so many more?

In [ ]:
# your code for the questions: tables, figures, anything you quote in the answers

---

## Task 8. The same PCA the other way round  
*0.15 points: 0.07 for the code (tests), 0.08 for the answers*

Task 7 went through the **covariance matrix**, $\mathbf{C} = \frac{1}{n-1}\mathbf{Z}^{\top}\mathbf{Z}$
($p \times p$). A second route goes through the **Gram matrix**, the dot products between rows
instead of between columns:

$$\mathbf{G} = \mathbf{Z}\mathbf{Z}^{\top} \quad (n \times n), \qquad \mathbf{G}\mathbf{u}_i = \mu_i \mathbf{u}_i .$$

Entry $G_{ab}$ is the dot product of minute $a$ with minute $b$: how alike two moments of the pump
are. From $\mathbf{G}$:

1. eigen-decompose with `eigh`, sort in descending order, and set the eigenvalues below zero to
   zero (the tail of a rank-deficient $\mathbf{G}$ is rounding noise of either sign);
2. the shares of variance $r_i = \mu_i / \sum_j \mu_j$;
3. the scores $\mathbf{P} = \mathbf{U}\sqrt{\mathbf{M}}$, $\sqrt{\mathbf{M}} = \mathrm{diag}(\sqrt{\mu_1},
   \dots, \sqrt{\mu_n})$: column $i$ of $\mathbf{U}$ times $\sqrt{\mu_i}$.

Why does it work? Write $\mathbf{Z}$ as its singular value decomposition,
$\mathbf{Z} = \mathbf{U}\mathbf{S}\mathbf{V}^{\top}$. Then

$$\mathbf{Z}^{\top}\mathbf{Z} = \mathbf{V}\mathbf{S}^{2}\mathbf{V}^{\top}, \qquad
\mathbf{Z}\mathbf{Z}^{\top} = \mathbf{U}\mathbf{S}^{2}\mathbf{U}^{\top}, \qquad
\mathbf{Z}\mathbf{V} = \mathbf{U}\mathbf{S}.$$

Both matrices are built from the same singular values, so the shares agree, and the scores of
Task 7 equal $\mathbf{U}\mathbf{S}$ up to the sign of each column.

**Write**

- `pca_via_gram(X)`: for an $n \times p$ array returns `(mu, ratio, scores)`: the $n$ clipped
  eigenvalues of $\mathbf{G}$ in descending order, the $n$ shares, and the $n \times n$ scores;
- on `sub = work[FEATS].sample(3000, random_state=7)`:
  `n_nonzero`, an `int`, the number of eigenvalues of $\mathbf{G}$ larger than $10^{-9}$ times the
  largest; and `memory_full_gb`, a `float`, the memory in GB ($10^9$ bytes) that $\mathbf{G}$ of
  float64 would take for all the rows of `work`.

In [ ]:
def pca_via_gram(X):
    ...


n_nonzero = ...
memory_full_gb = ...

In [ ]:
# Tests for Task 8 (public). They check what your code does, not how it is written.
# Run this cell after your code; an AssertionError says which check failed.


def test_both_routes_agree_on_a_demo():
    mixing = np.array([[3, 1, 0, 0], [0, 2, 0, 0], [0, 0, 1, 0], [0, 0, 0, 0.5]])
    demo = np.random.default_rng(0).normal(size=(200, 4)) @ mixing
    mu, ratio_g, scores_g = pca_via_gram(demo)
    Z = (demo - demo.mean(0)) / demo.std(0)
    lam = np.sort(np.linalg.eigvalsh(Z.T @ Z / 199))[::-1]

    message = "shapes n, n and n x n"
    assert mu.shape == (200,) and ratio_g.shape == (200,), message
    assert scores_g.shape == (200, 200), message

    message = "the first p shares are those of the covariance route, the rest are zero"
    first_p_agree = np.allclose(ratio_g[:4], lam / lam.sum())
    rest_zero = np.allclose(ratio_g[4:], 0, atol=1e-12)
    assert first_p_agree and rest_zero, message
    assert np.allclose(mu[:4] / lam, 199), "mu_i = (n - 1) lambda_i"


test_both_routes_agree_on_a_demo()


def test_scores_match_up_to_sign():
    demo = np.random.default_rng(2).normal(size=(80, 3)) * [3, 2, 1]
    mu, ratio_g, scores_g = pca_via_gram(demo)
    Z = (demo - demo.mean(0)) / demo.std(0)
    lam, V = np.linalg.eigh(Z.T @ Z)
    P = Z @ V[:, ::-1]
    same_up_to_sign = np.allclose(np.abs(scores_g[:, :3]), np.abs(P), atol=1e-8)
    assert same_up_to_sign, "scores U sqrt(mu) equal Z V up to sign"
    sorted_and_clipped = (mu >= 0).all() and np.all(np.diff(mu) <= 0)
    assert sorted_and_clipped, "eigenvalues clipped at zero, in descending order"


test_scores_match_up_to_sign()


def test_results_have_the_right_type():
    is_count = isinstance(n_nonzero, (int, np.integer)) and 1 <= n_nonzero <= 3000
    assert is_count, "n_nonzero: a count of eigenvalues"
    is_float = isinstance(memory_full_gb, float) and memory_full_gb > 0
    assert is_float, "memory_full_gb: a float"


test_results_have_the_right_type()


print("all tests passed")

**Questions** (answer them in `practice_1_answers.md`, Task 8, with the numbers your code printed):

1. Time both routes on `sub` and report the shape of the matrix each had to decompose.
2. Print the first five shares of both routes, the largest difference between them, $\mu_i/\lambda_i$ for the first five, and whether the first two score columns agree. Explain each agreement from the SVD.
3. How many eigenvalues of $\mathbf{G}$ are non-zero, and why that number? How much memory would $\mathbf{G}$ need for all the rows?
4. Where do the two routes differ: what does each hand you, at what cost? When would you choose the Gram route?

In [ ]:
# your code for the questions: tables, figures, anything you quote in the answers

---

### Sources

- Pump sensor data, Kaggle `nphantawee/pump-sensor-data` (the data).
- Tukey J.W. *Exploratory Data Analysis*. Addison-Wesley, 1977. (the IQR fences)
- Iglewicz B., Hoaglin D.C. *How to Detect and Handle Outliers*. ASQC, 1993. (the robust
  z-score and its 3.5)
- Jolliffe I.T. *Principal Component Analysis*, 2nd ed. Springer, 2002. (both routes to PCA)